# Nutrition50 Dish Metadata Extraction

Filter the full Nutrition5k cafe1 metadata down to the 50 dishes listed in the Nutrition50 imagery folder.


## Steps

1. Collect the dish IDs listed under the Nutrition50 imagery directory.
2. Load the `nutrition5k_dataset_metadata_dish_metadata_cafe1.csv` file.
3. Keep only the rows whose `dish_id` matches those 50 dishes.
4. Save the filtered table as `nutrition50_validation.csv` inside the FoodSAM folder.


In [ ]:
import csv
from pathlib import Path

import pandas as pd


In [ ]:
BASE_DIR = Path('.').resolve()
IMAGERY_DIR = (
    BASE_DIR
    / 'FoodSAM'
    / 'dataset'
    / 'nutrition50_subset-20251021T151433Z-1-001'
    / 'nutrition50_subset'
    / 'imagery'
)

METADATA_CSV = (
    BASE_DIR
    / 'FoodSAM'
    / 'dataset'
    / 'nutrition50_subset-20251021T151433Z-1-001'
    / 'nutrition50_subset'
    / 'metadata'
    / 'nutrition5k_dataset_metadata_dish_metadata_cafe1.csv'
)

OUTPUT_CSV = BASE_DIR / 'FoodSAM' / 'nutrition50_validation.csv'

print(f"Imagery directory located at: {IMAGERY_DIR}")
print(f"Metadata CSV located at: {METADATA_CSV}")
print(f"Filtered output will be written to: {OUTPUT_CSV}")


In [ ]:
def collect_dish_ids(directory: Path) -> set[str]:
    dish_ids: set[str] = set()
    if not directory.exists():
        raise FileNotFoundError(f"Imagery directory not found: {directory}")
    for entry in directory.iterdir():
        if entry.is_dir() and entry.name.startswith('dish_'):
            dish_ids.add(entry.name)
    return dish_ids


imagery_dish_ids = collect_dish_ids(IMAGERY_DIR)
print(f"Collected {len(imagery_dish_ids)} dish IDs from imagery.")
if len(imagery_dish_ids) != 50:
    print('Warning: expected 50 dishes in imagery; please verify the dataset.')


In [ ]:
with METADATA_CSV.open('r', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    filtered_rows = [row for row in reader if row and row[0] in imagery_dish_ids]

if len(filtered_rows) != len(imagery_dish_ids):
    print(
        f"Warning: filtered rows ({len(filtered_rows)}) do not match imagery dish IDs ({len(imagery_dish_ids)})."
    )

max_cols = max(len(row) for row in filtered_rows) if filtered_rows else 0
padded_rows = [row + [''] * (max_cols - len(row)) for row in filtered_rows]

filtered_df = pd.DataFrame(padded_rows)
filtered_df.sort_values(by=0, inplace=True)
filtered_df.to_csv(OUTPUT_CSV, index=False, header=False)

print(f"Filtered rows saved to {OUTPUT_CSV} (columns: {max_cols})")
filtered_df.head()
